In [2]:
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.client import MlflowClient
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import subprocess

mlflow.set_tracking_uri("http://localhost:5000")
_ = mlflow.set_experiment("Reproducibility_Drill")

In [ ]:
# Fetch the current git commit hash
try:
    git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).strip().decode('utf-8')
except Exception:
    git_commit = "unknown"
print(git_commit)

In [ ]:
df = pd.read_csv("Iris.csv")
print(df.head(5))
if 'Id' in df.columns:
    df = df.drop(columns=['Id']) # Drop Id if present to avoid skewing features

X = df.drop(columns=['Species'])
y = df['Species']

   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa


In [ ]:
seed = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

params = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": seed
}

In [ ]:
with mlflow.start_run() as run:
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    # Log parameters, metrics, seed, and git_commit tag
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_param("seed", seed)
    mlflow.set_tag("git_commit", git_commit)

    # Log artifact and register model
    signature = infer_signature(X_train, preds)
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        registered_model_name="Iris_RF_Classifier"
    )
    
    print(f"Logged run {run.info.run_id} with commit {git_commit}")

In [ ]:
client = MlflowClient()
latest_version = client.get_latest_versions("Iris_RF_Classifier", stages=["None"])[0].version

client.transition_model_version_stage(
    name="Iris_RF_Classifier",
    version=latest_version,
    stage="Staging"
)
print(f"Model version {latest_version} successfully transitioned to Staging.")